#  **02. Data Preprocessing & Feature Engineering**

## Tổng quan (Overview)
Notebook này tập trung vào việc chuẩn bị dữ liệu (Data Preparation) nối tiếp sau bước Khám phá dữ liệu (EDA). Mục đích nhằm biến đổi dữ liệu về trạng thái tối ưu nhất dành cho các mô hình Machine Learning.

- **Đầu vào (Input):** File dữ liệu thô hoặc đã qua làm sạch cơ bản.

- **Đầu ra (Output):** Bộ dữ liệu hoàn chuẩn bị cho mô hình học máy.

##  Bố cục Pipeline (Table of Contents)
* [0. Cài đặt Môi trường & Load dữ liệu](#0)
* [1. Tiền xử lý dữ liệu - Giai đoạn 1 (Làm sạch cơ sở & Hợp nhất)](#1)
    * [1.1 Bước 1: Quản trị không gian đặc trưng & biến định danh (Drop IDs, Name, Tách Target)](#1-1)
    * [1.2 Bước 2: Chuẩn hóa văn bản nhiễu trước khi Encode (Normalize Text)](#1-2)
    * [1.3 Bước 3: Hợp nhất đặc trưng MNAR (Thay thế Median Imputation)](#1-3)
* [2. Kỹ sư hóa đặc trưng (Feature Engineering)](#2) *(Thực thi ngay sau dòng MNAR)*
    * [2.1 FE-1: Tổng hợp ma trận áp lực đa chiều](#2-1)
    * [2.2 FE-2: Hệ số hao mòn sinh học & thâm hụt thời gian](#2-2)
    * [2.3 FE-3: Phân rã và xác định cờ nhóm tuổi nguy cơ cao](#2-3)
    * [2.4 FE-4: Cấu trúc điểm rủi ro lâm sàng phức hợp](#2-4)
    * [2.5 FE-5: Chỉ số hài lòng tổng hợp](#2-5)
    * [2.6 FE-6 (Tùy chọn): Khai thác NLP (TF-IDF & SVD) cho Profession & Degree](#2-6)
* [3. Tiền xử lý dữ liệu - Giai đoạn 2 (Xử lý Missing & Encoding)](#3)
    * [3.1 Bước 4: Xử lý giá trị thiếu (NaN) còn sót lại (Theo tư duy từng Model)](#3-1)
    * [3.2 Bước 5: Ordinal Encoding (Cho các biến có tính thứ tự)](#3-2)
    * [3.3 Bước 6: One-Hot Encoding (Cho các biến danh nghĩa <= 5 nhãn)](#3-3)
    * [3.4 Bước 7: Smoothed Target Encoding (Xử lý High-Cardinality với Cross-Val)](#3-4)
    * [3.5 Bước 8: Chuẩn hóa không gian độ lượng (Scaling độc lập cho Neural Networks)](#3-5)
* [4. Tối ưu Đánh giá và Mất cân bằng lớp (Validation & Class Imbalance)](#4)
    * [4.1 Bước 9: Phân rã Cross-Validation (chuẩn bị hạ tầng Stacking)](#4-1)
    * [4.2 Bước 10: Trọng số nội tại (Scale Pos Weight & Không gian Optuna)](#4-2)
* [5. Xuất và Lưu trữ dữ liệu chuẩn (Export to `.pkl` / `.csv`)](#5)


# 0. Cài đặt môi trường và load dữ liệu

In [1]:
pip install -r requirements.txt


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [7]:
import os 
import warnings
from pathlib import Path 
import numpy as np 
import pandas as pd 

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)   

#================ THIẾT LẬP ĐƯỜNG DẪN =======================
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục Root của Project: {PROJECT_ROOT}")


# ================= CẤU HÌNH CÁC THAM SỐ ====================
class Config:
    # --- Trạng thái Seed hệ thống (Đảm bảo Reproducibility) ---
    RANDOM_STATE = 42
    
    # --- Cấu hình Machine Learning ---
    N_FOLDS = 5                 # [Bước 9] Phân rã K-Fold Cross Validation
    TREE_MISSING_VAL = -1       # [Bước 4] Giá trị missing cho XGBoost/LightGBM
    NN_IMPUTE_NEIGHBORS = 5     # [Bước 4] k của KNNImputer cho mạng Neural
    
    # --- Cấu hình Tên cột cốt lõi ---
    TARGET_COL = "Depression"
    DROP_COLS = ["id", "Name"]  # [Bước 1] Các cột rác gây bùng nổ chiều
    
    # --- Cấu hình file Nguồn và Đích ---
    TRAIN_INPUT = RAW_DATA_DIR / "train.csv"
    TEST_INPUT = RAW_DATA_DIR / "test.csv"
    
    # Các file Output cuối cùng (Cho Bước 11)
    TRAIN_TREE_OUT = PROCESSED_DATA_DIR / "train_tree_ready.pkl"
    TEST_TREE_OUT = PROCESSED_DATA_DIR / "test_tree_ready.pkl"
    
    TRAIN_NN_OUT = PROCESSED_DATA_DIR / "train_nn_ready.pkl"
    TEST_NN_OUT = PROCESSED_DATA_DIR / "test_nn_ready.pkl"
config = Config()

# ============================= LOAD DATASET =====================
try:
    train_df = pd.read_csv(config.TRAIN_INPUT)
    test_df = pd.read_csv(config.TEST_INPUT)
    
    print("-" * 50)
    print(f"Tải dữ liệu thành công!")
    print(f"Tập Train (Row, Col): {train_df.shape}")
    print(f"Tập Test  (Row, Col): {test_df.shape}")
    print("-" * 50)
except FileNotFoundError as e:
    print(f"LỖI KHÔNG TÌM THẤY DỮ LIỆU. Vui lòng kiểm tra lại thư mục /data/raw! Chi tiết lỗi: {e}")



Thư mục Root của Project: d:\2025-2026 HKII\Data Analystic\Project\Exploring-Mental-Health-Data
--------------------------------------------------
Tải dữ liệu thành công!
Tập Train (Row, Col): (140700, 20)
Tập Test  (Row, Col): (93800, 19)
--------------------------------------------------
